In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import BertConfig, BertForSequenceClassification, BertTokenizer
from torch.optim import AdamW
from datasets import load_dataset
from tqdm.auto import tqdm

# 檢查並設定運算設備 (如果有 GPU/MPS 就使用)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"目前使用的運算設備: {device}")

目前使用的運算設備: cuda


In [2]:
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_LENGTH = 256

# 訓練資料筆數 (這裡調小一點供快速測試跑通，實戰時請拿掉這層限制)
TRAIN_SAMPLES = 25000
VAL_SAMPLES = 5000
NUM_LABELS = 2  # 情感分析 (正面/負面)

In [3]:
print("正在載入資料集...")
dataset = load_dataset("imdb")

# 取部分資料作為示範
train_dataset = dataset['train'].shuffle(seed=42).select(range(TRAIN_SAMPLES))
val_dataset = dataset['test'].shuffle(seed=42).select(range(VAL_SAMPLES))

tokenizer = BertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenization 處理函數
def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

print("正在進行文字 Tokenize ...")
train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

# 僅保留模型需要的 Tensor 欄位
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 建立 DataLoader
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print("資料處理完成！")

正在載入資料集...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/imdb@e6281661ce1c48d982bc483cf8a173c1bbeb5d31/.huggingface.yaml'. Repository id must be 'namespace/name', got 'imdb'.

In [ ]:
config = BertConfig(
    vocab_size=30522,            
    hidden_size=256,             
    num_hidden_layers=4,         
    num_attention_heads=8,       
    intermediate_size=1024,      
    max_position_embeddings=512, 
    num_labels=NUM_LABELS
)

# 建立模型 #bert-base-uncased, roberta-base, microsoft/deberta-v3-base, distilbert-base-uncased
model = BertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', 
    num_labels=NUM_LABELS
)
model.to(device)

# 定義優化器
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

print(f"模型已載入至 {device} 並準備就緒。")

You are using a model of type `distilbert` to instantiate a model of type `bert`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Loading weights: 0it [00:00, ?it/s]
BertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                                                                      | Status     | 
-------------------------------------------------------------------------+------------+-
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.q_lin.weight   | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.output_layer_norm.bias   | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.ffn.lin1.weight          | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.sa_layer_norm.bias      

模型已載入至 mps 並準備就緒。


In [ ]:
for epoch in range(EPOCHS):
    print(f"\n======== Epoch {epoch+1} / {EPOCHS} ========")
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_dataloader, desc="Training")
    for batch in progress_bar:
        # 1. 將資料移動到對應設備
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # 2. 梯度歸零
        model.zero_grad()
        
        # 3. 前向傳播 (Forward pass)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        
        # 4. 反向傳播 (Backward pass)
        loss.backward()
        
        # 5. 更新模型權重
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # 梯度裁剪防止爆炸
        optimizer.step()
        
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"平均訓練 Loss: {avg_train_loss:.4f}")


======== Epoch 1 / 3 ========


Training:   0%|          | 1/1563 [01:22<35:58:19, 82.91s/it, loss=0.7263]

In [ ]:
print("開始進行驗證...")
model.eval()
val_loss = 0
correct_predictions = 0
total_predictions = 0

with torch.no_grad(): # 推論階段不需要計算梯度以節省記憶體
    for batch in tqdm(val_dataloader, desc="Validating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        val_loss += loss.item()
        
        # 計算準確率
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        correct_predictions += (predictions == labels).sum().item()
        total_predictions += labels.size(0)

avg_val_loss = val_loss / len(val_dataloader)
accuracy = correct_predictions / total_predictions

print(f"\n💡 最終驗證結果：")
print(f"驗證 Loss: {avg_val_loss:.4f}")
print(f"準確率 (Accuracy): {accuracy * 100:.2f}%")